# Session 4: Design Patterns I — Creational & Structural

> Implement Factory, Singleton, Adapter, and Proxy patterns in production scenarios.

## 1. Singleton & Factory

### Singleton Pattern
**"Ensure a class has only one instance, and provide a global point of access to it."**

Use when the object represents a shared, expensive resource: a database connection pool, a config loader, a logging handler. The key implementation detail in Python is overriding `__new__` and storing the instance in a class variable. In threaded environments you must add a lock (see the additional examples).

**When NOT to use:** Don't use Singleton for simple configuration objects that could be passed as arguments — it makes code harder to test because you can't inject a replacement. Prefer dependency injection for testability, and use Singleton only for resources that are genuinely global by nature.

### Factory Pattern
**"Define an interface for creating an object, but let subclasses/config decide which class to instantiate."**

The key benefit: the code that *uses* a storage backend never needs to know whether it's S3, GCS, or Azure. You pass `'s3'` to `StorageFactory.create()` and get back the right object. This makes it trivial to add a new provider (add a class + register one line in the factory dict) without touching any existing call sites.

In [ ]:
# Singleton — single DB connection
class DatabasePool:
    _instance = None
    def __new__(cls):
        if not cls._instance:
            cls._instance = super().__new__(cls)
            cls._instance.conn = 'postgresql://prod-db:5432/app'
            print('🔌 Connection pool initialized')
        return cls._instance

db1, db2 = DatabasePool(), DatabasePool()
print(f'Same instance: {db1 is db2}')  # True

# Factory — decouple object creation
from abc import ABC, abstractmethod
class StorageBackend(ABC):
    @abstractmethod
    def upload(self, file): pass

class S3Backend(StorageBackend):
    def upload(self, file): return f's3://bucket/{file}'

class GCSBackend(StorageBackend):
    def upload(self, file): return f'gs://bucket/{file}'

class StorageFactory:
    _backends = {'s3': S3Backend, 'gcs': GCSBackend}
    @classmethod
    def create(cls, provider: str) -> StorageBackend:
        if provider not in cls._backends:
            raise ValueError(f'Unknown provider: {provider}')
        return cls._backends[provider]()

storage = StorageFactory.create('s3')
print(storage.upload('report.pdf'))

## 💡 Additional Examples: Singleton & Factory

**Example 1 — Thread-safe Singleton (AppConfig)**
The naive `if not cls._instance` check has a race condition: two threads can both pass the check simultaneously and create two instances. The fix is double-checked locking with `threading.Lock()`. Once the instance exists the lock is never acquired again, so there is no performance penalty on the hot path.

**Example 2 — Abstract Factory (UI Themes)**
A single `UIThemeFactory` abstraction hides whether you are building a `LightTheme` or `DarkTheme`. The calling code just calls `factory.create_button()` — it never mentions colours or hex codes. Switching themes is a one-line change at the composition root.

**Example 3 — Builder Pattern (EmailBuilder)**
The Builder pattern shines when constructing objects with many optional fields. Without it, you'd have a constructor like `Email(to, subject, body, cc=None, bcc=None, attachments=None, reply_to=None, ...)` which is hard to read at the call site. The fluent builder lets you call only the methods you need and end with `.build()` for a validated object.

In [ ]:
# ─── Example 1: Singleton — Thread-safe App Config ───────────────────────────
import threading

class AppConfig:
    """Thread-safe Singleton for application-wide configuration."""
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        if not cls._instance:
            with cls._lock:  # Double-checked locking prevents race conditions
                if not cls._instance:
                    cls._instance = super().__new__(cls)
                    cls._instance._settings = {
                        'max_upload_size_mb': 50,
                        'session_timeout_sec': 3600,
                        'feature_flags': {'dark_mode': True, 'beta_api': False},
                    }
                    print('⚙️  AppConfig initialized (once)')
        return cls._instance

    def get(self, key: str, default=None):
        return self._settings.get(key, default)

    def set(self, key: str, value):
        self._settings[key] = value

cfg1 = AppConfig()
cfg2 = AppConfig()
cfg1.set('feature_flags', {**cfg1.get('feature_flags'), 'beta_api': True})
print(f'Same instance: {cfg1 is cfg2}')
print(f'beta_api via cfg2: {cfg2.get("feature_flags")["beta_api"]}')  # True — shared state

# ─── Example 2: Abstract Factory — Multi-tenant UI Theme ─────────────────────
from abc import ABC, abstractmethod

class Button(ABC):
    @abstractmethod
    def render(self) -> str: ...

class InputField(ABC):
    @abstractmethod
    def render(self) -> str: ...

class ThemeFactory(ABC):
    @abstractmethod
    def create_button(self) -> Button: ...
    @abstractmethod
    def create_input(self) -> InputField: ...

# Light theme implementation
class LightButton(Button):
    def render(self): return '<button style="bg:white;color:black">Click</button>'

class LightInput(InputField):
    def render(self): return '<input style="bg:#f5f5f5;border:1px solid #ccc">'

class LightThemeFactory(ThemeFactory):
    def create_button(self): return LightButton()
    def create_input(self): return LightInput()

# Dark theme implementation
class DarkButton(Button):
    def render(self): return '<button style="bg:#1e1e1e;color:white">Click</button>'

class DarkInput(InputField):
    def render(self): return '<input style="bg:#2d2d2d;border:1px solid #555">'

class DarkThemeFactory(ThemeFactory):
    def create_button(self): return DarkButton()
    def create_input(self): return DarkInput()

def build_ui(factory: ThemeFactory):
    btn = factory.create_button()
    inp = factory.create_input()
    print(f'  Button: {btn.render()}')
    print(f'  Input:  {inp.render()}')

print('Light Theme:')
build_ui(LightThemeFactory())
print('Dark Theme:')
build_ui(DarkThemeFactory())

# ─── Example 3: Builder Pattern — Fluent Email Builder ───────────────────────
from dataclasses import dataclass, field

@dataclass
class Email:
    to: str = ''
    subject: str = ''
    body: str = ''
    cc: list = field(default_factory=list)
    attachments: list = field(default_factory=list)
    html: bool = False

class EmailBuilder:
    def __init__(self): self._email = Email()
    def to(self, recipient: str):       self._email.to = recipient;      return self
    def subject(self, s: str):          self._email.subject = s;          return self
    def body(self, b: str):             self._email.body = b;             return self
    def cc(self, *recipients):          self._email.cc = list(recipients); return self
    def attach(self, filename: str):    self._email.attachments.append(filename); return self
    def as_html(self):                  self._email.html = True;          return self
    def build(self) -> Email:
        if not self._email.to:      raise ValueError('Recipient is required')
        if not self._email.subject: raise ValueError('Subject is required')
        return self._email

email = (
    EmailBuilder()
    .to('client@example.com')
    .subject('Monthly Report — April 2026')
    .body('<h1>Report ready</h1>')
    .cc('manager@company.com')
    .attach('report_april_2026.pdf')
    .as_html()
    .build()
)
print(f'\n📧 Built Email: to={email.to}, cc={email.cc}, attachments={email.attachments}, html={email.html}')


## 2. Adapter & Proxy

### Adapter Pattern
**"Convert the interface of a class into another interface that clients expect."**

You have a legacy payment gateway with a method `process_cc(card_number, amount, currency)`. Your new unified interface expects `charge(amount: float, currency: str, token: str)`. Instead of rewriting every call site — or worse, rewriting the legacy code — you write a thin `LegacyPaymentAdapter` that translates between them. The adapter is the only class that knows about both interfaces.

### Proxy Pattern
**"Provide a surrogate or placeholder for another object to control access to it."**

A proxy wraps an object and adds behaviour *around* every call: logging, caching, rate limiting, auth checks. The caller is completely unaware — it holds a reference typed as the interface and calls the same methods. A `LoggingProxy` that records timing is one of the most practical uses: you can wrap any object without modifying it, and remove the proxy in production with zero code change.

In [ ]:
# Proxy — add auth + caching layer
import time
class APIClient:
    def fetch(self, endpoint): return {'data': f'result from {endpoint}', 'ts': time.time()}

class SecureAPIProxy:
    def __init__(self, client: APIClient, token: str):
        self._client = client
        self._token = token
        self._cache = {}
    def fetch(self, endpoint):
        if not self._token: raise PermissionError('Unauthorized')
        if endpoint in self._cache:
            print(f'Cache hit: {endpoint}')
            return self._cache[endpoint]
        result = self._client.fetch(endpoint)
        self._cache[endpoint] = result
        return result

proxy = SecureAPIProxy(APIClient(), token='Bearer xyz')
print(proxy.fetch('/users'))  # API call
print(proxy.fetch('/users'))  # Cache hit

## 💡 Additional Examples: Adapter & Proxy

**Example 1 — Adapter: Legacy Payment Gateway**
The real-world scenario: you integrate a third-party payment SDK from 2015. It has its own method signatures, error types, and return formats. Rather than spreading `legacy_sdk.process_cc(...)` calls throughout your codebase, you create one adapter class. If you ever replace the SDK, you rewrite only the adapter — no other file changes.

**Example 2 — Logging Proxy**
The proxy wraps any object that has a `process` method and automatically logs the method name, arguments, return value, and execution time. Because it respects the same interface, you can swap `service = LoggingProxy(RealService())` for `service = RealService()` with a one-line change. This is how APM tools and ORMs instrument your code transparently.

**Example 3 — Decorator chain for middleware**
Python function decorators are a lightweight form of the Decorator structural pattern. Stacking `@require_auth` and `@log_request` on a route handler means each concern is written once and composed freely — exactly what OCP and SRP recommend.

In [ ]:
# ─── Example 1: Adapter — Legacy Payment Gateway ─────────────────────────────
# Legacy system returns old format (XML-like dict)
class LegacyPaymentGateway:
    def make_payment(self, card_no: str, exp: str, amount_cents: int) -> dict:
        return {'status_code': 0, 'msg': 'OK', 'txn_ref': 'TXN-9988'}

# New standard interface expected by the application
from abc import ABC, abstractmethod
class PaymentGateway(ABC):
    @abstractmethod
    def charge(self, token: str, amount: float) -> dict: ...

# Adapter wraps legacy gateway to expose the new interface — no changes to old code
class LegacyGatewayAdapter(PaymentGateway):
    def __init__(self, legacy: LegacyPaymentGateway):
        self._legacy = legacy

    def charge(self, token: str, amount: float) -> dict:
        # Map new interface → old interface
        result = self._legacy.make_payment(
            card_no=token,
            exp='12/28',                    # Retrieved from token vault in real usage
            amount_cents=int(amount * 100)
        )
        # Map old response format → new response format
        return {
            'success': result['status_code'] == 0,
            'transaction_id': result['txn_ref'],
            'message': result['msg'],
        }

gateway = LegacyGatewayAdapter(LegacyPaymentGateway())
result = gateway.charge('tok_visa_4242', 99.99)
print(f'Payment result: {result}')

# ─── Example 2: Proxy — Transparent Logging ──────────────────────────────────
import time
from collections import defaultdict

class DataService:
    def get_user_data(self, user_id: int) -> dict:
        return {'id': user_id, 'name': f'User_{user_id}', 'email': f'user{user_id}@example.com'}

class LoggingProxy:
    """Adds logging around DataService without modifying it."""
    def __init__(self, service: DataService):
        self._service = service
        self._calls = []

    def get_user_data(self, user_id: int) -> dict:
        start = time.perf_counter()
        result = self._service.get_user_data(user_id)
        elapsed_ms = (time.perf_counter() - start) * 1000
        log_entry = {'method': 'get_user_data', 'args': {'user_id': user_id}, 'ms': round(elapsed_ms, 2)}
        self._calls.append(log_entry)
        print(f'📝 LOG: {log_entry}')
        return result

    def get_call_summary(self):
        return {'total_calls': len(self._calls), 'calls': self._calls}

proxy = LoggingProxy(DataService())
proxy.get_user_data(1)
proxy.get_user_data(2)
proxy.get_user_data(1)
print(f'\n📊 Summary: {proxy.get_call_summary()["total_calls"]} calls made')

# ─── Example 3: Decorator Pattern — Request Middleware Chain ─────────────────
from functools import wraps

def require_auth(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        token = kwargs.get('token') or (args[1] if len(args) > 1 else None)
        if not token or not token.startswith('Bearer '):
            raise PermissionError('Valid Bearer token required')
        return func(*args, **kwargs)
    return wrapper

def log_request(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f'→ {func.__name__} called with args={args[1:]}, kwargs={kwargs}')
        result = func(*args, **kwargs)
        print(f'← {func.__name__} returned: {result}')
        return result
    return wrapper

class OrderController:
    @log_request
    @require_auth
    def get_order(self, token: str, order_id: int) -> dict:
        return {'id': order_id, 'status': 'shipped', 'total': 150.0}

ctrl = OrderController()
try:
    ctrl.get_order('no-token', 42)           # Raises PermissionError
except PermissionError as e:
    print(f'❌ {e}')

ctrl.get_order('Bearer valid_jwt_xyz', 42)   # OK


Payment result: {'success': True, 'transaction_id': 'TXN-9988', 'message': 'OK'}

📝 LOG: {'method': 'get_user_data', 'args': {'user_id': 1}, 'ms': 0.01}
📝 LOG: {'method': 'get_user_data', 'args': {'user_id': 2}, 'ms': 0.01}
📝 LOG: {'method': 'get_user_data', 'args': {'user_id': 1}, 'ms': 0.18}

📊 Summary: 3 calls made

## 3. Practice

### 🧪 AI Lab / Practice

> **TODO:** Implement the Proxy pattern for a rate-limited external API client: max 5 requests/minute per endpoint. If limit exceeded, raise RateLimitError with retry-after seconds. Write 3 unit tests covering: normal flow, cache hit, and rate limit enforcement.

In [ ]:
# TODO: Implement your solution here
import time
import unittest
from collections import defaultdict

class RateLimitError(Exception):
    def __init__(self, retry_after: int):
        self.retry_after = retry_after
        super().__init__(f'Too many requests. Retry after {retry_after} seconds')

class APIClient:
    def fetch(self, endpoint: str) -> dict:
        return {'status': 'success', 'endpoint': endpoint, 'data':'some_data'}
    
class RateLimitedAPIProxy:
    def __init__(self, real_client: APIClient):
        self._client = real_client
        self._max_requests = 5
        self._window_seconds = 60
        self._request_history = defaultdict(list)

    def fetch(self, endpoint: str) -> dict:
        current_time = time.time()
        history = self._request_history[endpoint]

        self._request_history[endpoint] = [
            t for t in history if current_time - t < self._window_seconds
        ]
        active_history = self._request_history[endpoint]

        if len(active_history) >= self._max_requests:
            oldest_request_time = active_history[0]
            retry_after = int(self._window_seconds - (current_time - oldest_request_time))
            retry_after = max(retry_after, 0)
            raise RateLimitError(retry_after=retry_after)
        self._request_history[endpoint].append(current_time)
        return self._client.fetch(endpoint)
    
# ─── UNIT TESTS ──────────────────────────────────────────────────────────────
class TestRateLimitedAPIProxy(unittest.TestCase):
    
    def setUp(self):
        # Chạy trước mỗi test case để làm mới client và proxy
        self.real_client = APIClient()
        self.proxy = RateLimitedAPIProxy(self.real_client)

    def test_normal_flow(self):
        """Test 1: Kiểm tra luồng chạy bình thường dưới hạn mức"""
        # Gọi 3 lần liên tiếp (vẫn dưới mức 5)
        for i in range(3):
            response = self.proxy.fetch("/users")
            self.assertEqual(response["status"], "success")
            self.assertEqual(response["endpoint"], "/users")

    def test_endpoint_isolation(self):
        """Test 2: Kiểm tra việc đếm request độc lập giữa các endpoint khác nhau"""
        # Gọi endpoint A 5 lần -> chạm mốc tối đa
        for _ in range(5):
            self.proxy.fetch("/users")
            
        # Endpoint B vẫn phải gọi được bình thường vì nó chưa bị tính request nào
        try:
            response = self.proxy.fetch("/products")
            self.assertEqual(response["status"], "success")
        except RateLimitError:
            self.fail("Endpoint /products bị block nhầm bởi giới hạn của /users")

    def test_rate_limit_enforcement(self):
        """Test 3: Kiểm tra việc kích hoạt chặn và ném lỗi RateLimitError khi vượt ngưỡng"""
        # Gọi 5 lần hợp lệ đầu tiên
        for _ in range(5):
            self.proxy.fetch("/analytics")
            
        # Lần thứ 6 chắc chắn phải bị chặn và ném ra đúng lỗi
        with self.assertRaises(RateLimitError) as context:
            self.proxy.fetch("/analytics")
            
        # Kiểm tra xem thuộc tính retry_after có tồn tại và hợp lệ không (từ 1 đến 60 giây)
        self.assertTrue(1 <= context.exception.retry_after <= 60)

# Chạy test nếu file này được thực thi trực tiếp
if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

In [ ]:
"""
Hướng dẫn:
  - Đọc kỹ từng đoạn CODE GỐC và các CÂU HỎI bên dưới.
  - Viết câu trả lời / code implement vào phần TODO.
  - Chạy file để kiểm tra kết quả.
"""
# BÀI 4: Singleton — Thread-safe AppConfig
# 
# ── CODE GỐC ──────────────────────────────────────────────────────
class AppConfig_Bad:
    _instance = None

    def __new__(cls):
        if not cls._instance:               # ← Race condition ở đây
            cls._instance = super().__new__(cls)
            cls._instance.settings = {'debug': True, 'db': 'postgres://prod:5432/app'}
        return cls._instance

    def get(self, key: str):
        return self.settings.get(key)

    def set(self, key: str, value):
        self.settings[key] = value

# ── CÂU HỎI ───────────────────────────────────────────────────────
# 1. Mô tả chính xác cơ chế race condition xảy ra ở AppConfig_Bad
#    trong môi trường multi-threading. Kết quả len(set(results))
#    có thể là bao nhiêu trong tình huống xấu nhất?
#
# 2. "Double-checked locking" nghĩa là gì? Tại sao phải kiểm tra
#    cls._instance HAI LẦN (một lần ngoài lock, một lần trong lock)?
#
# 3. Fix lại AppConfig thành thread-safe Singleton.
#    Viết demo chứng minh 100 thread đồng thời tạo ra đúng 1 instance.

# ── TODO ──────────────────────────────────────────────────────────
'''
1. Race condition xảy ra khi nhiều thread cùng kiểm tra điều kiện 'if not cls._instance'
và '_instance' đang là 'None'. Do đó, tất cả các thread này sẽ tạo instance mới,
dẫn đến việc tạo ra nhiều instance khác nhau thay vì chỉ một instance.
Trong tình huống xấu nhất, nếu có N threads cùng chạy đồng thời,
có thể tạo ra N instance khác nhau, do đó len(set(results)) có thể là N.

2. "Double-checked locking" là một kỹ thuật để đảm bảo rằng chỉ có một instance được tạo ra trong môi trường đa luồng.
Khi một thread kiểm tra 'if not cls._instance' và thấy rằng '_instance' đang là 'None', nó sẽ tiến hành tạo instance mới.
Tuy nhiên, nếu có nhiều thread cùng kiểm tra điều kiện này đồng thời,
tất cả chúng sẽ thấy '_instance' là 'None' và ác nhau.sẽ tạo ra nhiều instance kh
Để tránh điều này, chúng ta sử dụng một lock để đảm bảo rằng chỉ có một thread được phép tạo instance tại một thời điểm.
'''
#3. Fix lại AppConfig thành thread-safe Singleton.
import threading

class Appconfig_Good:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        if not cls._instance:
            with cls._lock:
                if not cls._instance:
                    cls._instance = super().__new__(cls)
                    cls._instance.settings = {'debug' : True, 'db': 'postgres://prod:5432/app'}
                    print('AppConfig_Good initialized')
        return cls._instance
    
    def get(self, key: str):
        return self.settings.get(key)
    
    def set(self, key: str, value):
        self.settings[key] = value

# Demo chứng minh 100 thread đồng thời tạo ra đúng 1 instance
def create_instance(results, index):
    instance = Appconfig_Good()
    results[index] = id(instance)
    
num_threads = 100
threads = []
results = [None] * num_threads
for i in range(num_threads):
    t = threading.Thread(target=create_instance, args=(results, i))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

unique_instances = set(results)
print(f'Number of unique instances created: {len(unique_instances)}')  # Should be 1


In [ ]:
# ══════════════════════════════════════════════════════════════════
# BÀI 5
# ══════════════════════════════════════════════════════════════════

# ── CODE GỐC ──────────────────────────────────────────────────────
# Ba service bên thứ 3 / legacy với interface khác nhau:

class SMSSender:
    def send_text(self, phone_number: str, body: str) -> bool:
        print(f"  [SMS] → {phone_number}: {body[:160]}")
        return True


class PushNotificationService:
    def push(self, device_token: str, title: str, payload: dict) -> None:
        print(f"  [PUSH] → {device_token}: {title} | {payload}")


class SlackClient:
    def post_message(self, channel: str, blocks: list) -> dict:
        print(f"  [SLACK] → #{channel}: {blocks[0]['text']}")
        return {'ok': True}


# Application chỉ biết interface thống nhất này:
class Notifier(ABC):
    @abstractmethod
    def notify(self, recipient: str, message: str) -> None: ...

# ── CÂU HỎI ───────────────────────────────────────────────────────
# 1. Pattern nào cần áp dụng ở đây? Tại sao không thể sửa trực tiếp
#    SMSSender, PushNotificationService, SlackClient?
#    (Gợi ý: nghĩ đến nguồn gốc của các class này)
#
# 2. Adapter khác gì Wrapper thông thường?
#    Adapter giải quyết vấn đề về "interface incompatibility" như thế nào?
#   
# 3. Implement 3 Adapter class: SMSNotifierAdapter, PushNotifierAdapter,
#    SlackNotifierAdapter. Đảm bảo application code chỉ gọi notify().
#    Viết NotificationDispatcher nhận list[Notifier] và broadcast.

# ── TODO ──────────────────────────────────────────────────────────
'''
1. Pattern cần áp dụng là Adapter pattern. Không thể sửa trực tiếp vì Ba service này của bên thứ 3 / Legacy
2. Adapter khác với Wrapper thông thường ở chỗ:
Adapter được thiết kế để làm cho một interface không tương thích trở nên tương thích với interface mà client mong đợi.
Adapter giải quyết vấn đề về giao diện không tương thích bằng cách đóng gói một đối tượng có interface không tương thích
và cung cấp một interface mới mà client có thể sử dụng.
Trong khi đó, Wrapper thường chỉ đơn giản là bao bọc một đối tượng để thêm chức năng mà không thay đổi interface của nó.
'''
#3. Implement 3 Adapter class: SMSNotifierAdapter, PushNotifierAdapter, SlackNotifierAdapter


class SMSSender:
    def send_text(self, phone_number: str, body: str) -> bool:
        print(f"  [SMS] → {phone_number}: {body[:160]}")
        return True


class PushNotificationService:
    def push(self, device_token: str, title: str, payload: dict) -> None:
        print(f"  [PUSH] → {device_token}: {title} | {payload}")


class SlackClient:
    def post_message(self, channel: str, blocks: list) -> dict:
        print(f"  [SLACK] → #{channel}: {blocks[0]['text']}")
        return {'ok': True}
    
from abc import ABC, abstractmethod

class Notifier(ABC):
    @abstractmethod
    def notify(self, recipient: str, message: str) -> None: ...

class SMSNotifierAdapter(Notifier):
    def __init__(self, sms_sender: SMSSender):
        self._sms_sender = sms_sender

    def notify(self, recipient: str, message: str) -> None:
        self._sms_sender.send_text( 
            phone_number = recipient,
            body = message
        )

class PushNotifierAdapter(Notifier):
    def __init__(self, push_service: PushNotificationService):
        self._push_service = push_service

    def notify(self, recipient: str, message: str) -> None:
        self._push_service.push(
            device_token = recipient,
            title = "Notification",
            payload = {"body": message}
        )

class SlackNotifierAdapter(Notifier):
    def __init__(self, slack_client: SlackClient):
        self._slack_client = slack_client

    def notify(self, recipient: str, message: str) -> None:
        blocks = [{'text': message,}]
        self._slack_client.post_message(
            channel = recipient,
            blocks = blocks
        )

class NotificationDispatcher:
    def __init__(self, notifiers: list[Notifier]):
        self._notifiers = notifiers

    def broadcast(self, recipient: str, message: str):
        for notifier in self._notifiers:
            notifier.notify(recipient, message)

# Demo
sms_adapter = SMSNotifierAdapter(SMSSender())
push_adapter = PushNotifierAdapter(PushNotificationService())
slack_adapter = SlackNotifierAdapter(SlackClient())
dispatcher = NotificationDispatcher([sms_adapter, push_adapter, slack_adapter])
dispatcher.broadcast("user123", "Your order has been shipped!")